# Walmart B2B — Node Level Cost Pricing

Orchestrator notebook for the full NLC pricing pipeline.

**Steps:**
1. Set parameters & flags
2. Load data
3. Run NLC model (two-pass: min_units=8, then 4)
4. Apply pricing rules
5. Build new DSV
6. Update tests tracker
7. Save outputs
8. Upload DSV to hybris

**Optional steps** (toggle via flags):
- Rollback handling
- National price updates
- Hybris upload

**Post-upload** (run separately, ~3 hours after hybris upload):
- FTP response validation

# Params

In [ ]:
import pandas as pd

# ── Core parameters ─────────────────────────────────────────────────
date_str = pd.to_datetime('today').strftime('%Y-%m-%d')
test = False
save = True
test_output = True                  # Save all outputs to local outputs/ instead of shared drive

# ── Optional steps ──────────────────────────────────────────────────
apply_rollbacks = True              # Remove rollback SKUs from NLC + apply RB prices to national
update_national_prices = False      # Override national prices from Excel file
upload_to_hybris = False            # Upload DSV to hybris after saving (requires Selenium/Chrome)

# ── Margin test filter (set to None to include all) ─────────────────
margin_test_start_dates = None      # e.g. ["2026-03-12"]

# ── Rollbacks path (changes monthly — update each cycle) ────────────
rollbacks_path = r"G:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_03 Rollbacks\2026-01 RBs\Approved RBs\Approved RBs start date 2026-02-01.xlsx"

# ── National price update config (only used if update_national_prices=True)
national_prices_path = None         # e.g. r"G:\...\Walmart B2B Item Report 2.20.26.xlsx"
national_prices_sheet = "National prices"
national_prices_skip_rows = 2

print(f"Date: {date_str}")
print(f"Test mode: {test}")
print(f"Test output (local): {test_output}")
print(f"Apply rollbacks: {apply_rollbacks}")
print(f"Rollbacks file: {rollbacks_path}")
print(f"Update national prices: {update_national_prices}")
print(f"Upload to hybris: {upload_to_hybris}")

# Setup

In [2]:
import sys
import os
import logging

import numpy as np

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
)

from src.data.loader import DataLoader
from src.models.nlc_model import NLCModel
from src.rules.pricing_rules import PricingRulesEngine
from src.dsv.dsv_builder import DSVBuilder
from src.tracker.tracker_updater import TrackerUpdater

print(f"Project root: {project_root}")

Project root: c:\Users\valen\Desktop\WalmartPricing


In [ ]:
from src.adapters.module_loader import load_yaml

if test_output:
    outputs_dir = os.path.join(project_root, "outputs")
    os.makedirs(outputs_dir, exist_ok=True)
    dsv_output_path = os.path.join(outputs_dir, f"New WalmartB2B DSV to upload {date_str}.csv")
    tracker_output_path = os.path.join(outputs_dir, f"Final node level costs tracker.csv")
    tracker_backup = False
    print(f"Test output mode: all files → {outputs_dir}")
else:
    dsv_output_path = None       # DSVBuilder will use shared drive default
    tracker_output_path = None   # TrackerUpdater will use shared drive default
    tracker_backup = True
    nlc_config = load_yaml("settings.yaml")
    print(f"Production mode: files → {nlc_config['shared_paths']['nlc_folder']}")

# Step 1-2: Load data & run NLC model

In [3]:
loader = DataLoader()
model = NLCModel(date_str=date_str)
model.load_data(loader, rollbacks_path=rollbacks_path)
df_output = model.run()

print(f"NLC output: {len(df_output)} SKU-Node rows")
display(df_output["Final target"].value_counts())
display(df_output["current_nlc_margin category"].value_counts())

2026-03-18 16:53:49,701 [INFO] src.models.nlc_model: Loading NLC model data for date=2026-03-18


2026-03-18 16:53:50,699 [INFO] src.adapters.google_api_adapter: Initializing Google API service connection...
2026-03-18 16:53:50,744 [INFO] googleapiclient.discovery_cache: file_cache is only supported with oauth2client<4.0.0
2026-03-18 16:53:50,751 [INFO] googleapiclient.discovery_cache: file_cache is only supported with oauth2client<4.0.0
2026-03-18 16:53:50,754 [INFO] src.adapters.google_api_adapter: Google API service connected.
2026-03-18 16:53:53,747 [INFO] src.data.loader: Max DSV date: 2026-03-18 00:00:00
2026-03-18 16:53:53,748 [INFO] src.data.loader: Reading DSV file: dsvpriceupload-walmartb2b-2026-03-18_06-01-11.csv


Download 86.
Download 100.


2026-03-18 16:54:02,050 [INFO] src.data.loader: Loading source: warehouse_node_mapping (type=folder)


Last modified time 2026-03-18T10:00:33.438Z
Download 100.


...

Download 100.
Last modified time 2026-03-18T10:01:24.060Z
Download 95.
Download 100.


2026-03-18 16:57:31,646 [INFO] src.models.nlc_model: Inventory loaded for 2026-03-18: 2193448 rows
2026-03-18 16:57:34,820 [INFO] src.models.nlc_model: Total inventory rows (excl rollbacks): 15516908
2026-03-18 16:57:34,861 [INFO] src.data.loader: Loading source: tests_tracker (type=local)
c:\Users\valen\Desktop\WalmartPricing\src\data\loader.py:150: DtypeWarning: Columns (0: Final price category, 1: Final price change category final, 2: Category inventory, 3: Sub-group, 4: Is in stock?, 5: Current walmart margin at NLC category, 6: sku_sales_category, 7: current_nlc_margin_date, 8: Last price update) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, **kwargs)
2026-03-18 16:57:53,713 [INFO] src.models.nlc_model: All NLC model data loaded.
2026-03-18 16:57:53,725 [INFO] src.models.nlc_model: Running NLC model...
2026-03-18 16:58:56,614 [INFO] src.models.nlc_model: NLC calc (min_units=8): 1576798 total, 1500747 workable
2026-03-18 16:58:

NLC output: 2270767 SKU-Node rows


Final target
Added                       726782
Margin test                 373118
Normal strategy             325539
Updated                     289018
No sales test               203228
Decreased - margin > 20%    129048
Updates test                102773
Wm margin split test         91955
Shipping cost added          25024
Less sales back               2058
Less sales                    2035
Name: count, dtype: int64

current_nlc_margin category
10.8% to <11.2%    724175
11.2% to <13%      375754
8% to <10.8%       288265
13% to <15%        272773
6% to <8%          162044
15% to <17%        160311
17% to <20%        134938
>=20%              113310
0% to <6%           38361
<0%                   836
Name: count, dtype: int64

## New SKU-Nodes + checks

In [4]:
df_new = df_output[df_output["New NLC"] == "Yes"].copy()
print(f"New SKU-Nodes: {len(df_new)}")
display(df_new["Min units"].value_counts())
display(df_new["Brand code"].value_counts().head(10))
display(df_new["Identifier"].value_counts().head(10))

New SKU-Nodes: 0


Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)

## Update current NLC margins & stock status

In [5]:
df_current_tests = model.df_current_tests.copy()
print(f"Tracker entries: {len(df_current_tests)}")
display(df_current_tests["Final target"].value_counts())

Tracker entries: 3281314


Final target
Added                       1261444
Normal strategy              496537
Margin test                  430874
Updated                      361799
No sales test                290673
Updates test                 147495
Decreased - margin > 20%     133267
Wm margin split test          95983
Shipping cost added           31921
Less sales                    15808
Less sales back               15513
Name: count, dtype: int64

# Step 3: Pricing rules

In [6]:
engine = PricingRulesEngine(
    df_output, model.df_current_tests, date_str, test_mode=test
)

## Walmart margin split test update

In [7]:
df_wm_split_dsv, df_wm_split_tracker = engine.get_wm_margin_split_updates()
print(f"Wm margin split DSV updates: {len(df_wm_split_dsv)}")
if len(df_wm_split_dsv) > 0:
    display(df_wm_split_dsv.head())

2026-03-18 17:00:50,851 [INFO] src.rules.pricing_rules: Wm margin split updates: 120 SKU-Nodes


Wm margin split DSV updates: 120


,SKU,Source,Price,SKU-Node
1534112,BRID0851725565T,6283261153,246.07,BRID0851725565T-6283261153
1534115,BRID0851725565T,6283261270,246.07,BRID0851725565T-6283261270
1534121,BRID0851725565T,6283261469,246.07,BRID0851725565T-6283261469
1534129,BRID0851725565T,628326695,246.07,BRID0851725565T-628326695
1536388,BRID3091822545W,6283261339,264.04,BRID3091822545W-6283261339


## Brands margin test update

In [8]:
df_margin_dsv, df_margin_tracker = engine.get_margin_test_updates(
    start_dates=margin_test_start_dates
)
print(f"Margin test DSV updates: {len(df_margin_dsv)}")
if len(df_margin_dsv) > 0:
    display(df_margin_dsv.head())

2026-03-18 17:00:53,761 [INFO] src.rules.pricing_rules: Margin test updates: 74614 SKU-Nodes


Margin test DSV updates: 74614


,SKU,Source,Price,SKU-Node
22530,ARMS0011316570T,62832605,40.49,ARMS0011316570T-62832605
22531,ARMS0011316570T,6283261045,40.49,ARMS0011316570T-6283261045
22558,ARMS0011418565H,628326855,33.71,ARMS0011418565H-628326855
22585,ARMS0011517560H,6283261264,40.10,ARMS0011517560H-6283261264
22591,ARMS0011518565H,6283261264,47.68,ARMS0011518565H-6283261264


## Low price updates

In [9]:
df_low_dsv, df_low_tracker = engine.get_low_price_updates()
print(f"Low price DSV updates: {len(df_low_dsv)}")
if len(df_low_dsv) > 0:
    display(df_low_tracker["Category inventory"].value_counts())
    display(df_low_tracker["Min units"].value_counts())

2026-03-18 17:00:56,032 [INFO] src.rules.pricing_rules: Low price updates: 532 SKU-Nodes


Low price DSV updates: 532


Category inventory
Not showing inventory    268
Below 6% margin          264
Name: count, dtype: int64

Min units
4    521
8     11
Name: count, dtype: int64

## High price updates

In [10]:
df_high_dsv, df_high_tracker = engine.get_high_price_updates()
print(f"High price DSV updates: {len(df_high_dsv)}")
if len(df_high_tracker) > 0:
    summary = (
        df_high_tracker.groupby("Final target")
        .agg(
            count=("Final target", "count"),
            avg_delta=("Final delta vs current %", "mean"),
        )
        .reset_index()
    )
    display(summary)

2026-03-18 17:00:56,238 [INFO] src.rules.pricing_rules: High price updates: 248 SKU-Nodes


High price DSV updates: 248


,Final target,count,avg_delta
0,Decreased - margin > 20%,248,-0.052449


## New SKU-Nodes

In [11]:
df_new_nodes_dsv, df_new_nodes_tracker = engine.get_new_sku_nodes()
print(f"New SKU-Nodes for DSV: {len(df_new_nodes_dsv)}")
if len(df_new_nodes_tracker) > 0:
    display(df_new_nodes_tracker["Final price category"].value_counts())

2026-03-18 17:00:56,418 [INFO] src.rules.pricing_rules: New SKU-Nodes: 0


New SKU-Nodes for DSV: 0


# Step 4: Build new DSV

In [12]:
builder = DSVBuilder(
    df_curr_dsv_original=model.df_curr_dsv_original,
    today_str=date_str,
)

# Start from current DSV
df_dsv_start = builder._prepare_starting_dsv()
print(f"Starting DSV: {len(df_dsv_start)} rows")

Starting DSV: 3288144 rows


## [Optional] Update national prices

Toggle: `update_national_prices` flag in Params cell.

In [13]:
if update_national_prices:
    print(f"Updating national prices from: {national_prices_path}")
    df_dsv_start = builder.apply_national_price_updates(
        df_dsv_start,
        national_prices_path=national_prices_path,
        sheet_name=national_prices_sheet,
        skip_rows=national_prices_skip_rows,
    )
else:
    print("[Skipped] Update national prices")

[Skipped] Update national prices


## [Optional] Rollback handling

Toggle: `apply_rollbacks` flag in Params cell.

When enabled:
- Removes rollback SKUs from NLC rows
- Applies rollback prices to national rows

In [14]:
if apply_rollbacks:
    print(f"Active rollbacks: {len(model.df_rollbacks)} rows, "
          f"{model.df_rollbacks['Product Code'].nunique()} unique SKUs")
    df_dsv_start = builder.apply_rollbacks(df_dsv_start, model.df_rollbacks)
else:
    print("[Skipped] Rollback handling")

Active rollbacks: 667 rows, 577 unique SKUs


2026-03-18 17:00:58,084 [INFO] src.dsv.dsv_builder: Rollbacks: removed 0 NLC rows for 577 rollback SKUs


## Apply NLC updates & build final DSV

In [15]:
list_dsv_updates = [df_wm_split_dsv, df_margin_dsv, df_low_dsv, df_high_dsv]

df_new_dsv = builder.build_from(
    df_dsv_start,
    list_dsv_updates=list_dsv_updates,
    df_new_nodes=df_new_nodes_dsv,
)

print(f"Final DSV: {len(df_new_dsv)} rows")
df_new_dsv.head()

2026-03-18 17:00:59,174 [INFO] src.dsv.dsv_builder: Starting DSV: 3288144 rows
2026-03-18 17:00:59,179 [INFO] src.dsv.dsv_builder: Original: 3288144 | Updates: 75514 | New nodes: 0 | Expected final: 3288144
2026-03-18 17:01:01,100 [INFO] src.dsv.dsv_builder: New DSV built: 3288144 rows


Final DSV: 3288144 rows


,SKU,Price,Minimum margin,Source
0,IRON0061417565H,40.11,4%,62832635
1,IRON0081417565H,40.11,4%,628326503
2,IRON0061417565H,40.11,4%,628326106
3,IRON0061417565H,40.11,4%,6283261141
4,APLS0031518555V,40.11,4%,628326916


## DSV validation

In [16]:
df_validation = builder.validate(df_new_dsv)

display(df_validation["Price change category"].value_counts())

df_changes = df_validation[df_validation["Price change category"] != "No change"]
df_changes["Is national?"] = np.where(
    df_changes["Source"] == "National", "Yes", "No"
)
display(df_changes["Is national?"].value_counts())
df_changes.head(10)

2026-03-18 17:01:06,490 [INFO] src.dsv.dsv_builder: DSV validation — changes: 75509
2026-03-18 17:01:06,782 [INFO] src.dsv.dsv_builder:   {'No change': 3212635, 'Increase': 38360, 'Decrease': 37149}


Price change category
No change    3212635
Increase       38360
Decrease       37149
Name: count, dtype: int64

Is national?
No     75494
Yes       15
Name: count, dtype: int64

,SKU,Price,Minimum margin,Source,SKU-Node,walmart_dsv_price,Price change,Price change category,Is national?
3152460,ZEET0251621560HXL,48.62,4%,National,ZEET0251621560HXL-National,45.29,3.33,Increase,Yes
3153882,COSM0061621560V,55.82,4%,National,COSM0061621560V-National,48.55,7.27,Increase,Yes
3154179,SAIL0171723565T,91.50,4%,National,SAIL0171723565T-National,83.32,8.18,Increase,Yes
3154518,TRAV0011523575S,61.77,4%,National,TRAV0011523575S-National,61.08,0.69,Increase,Yes
3154948,COSM0091722565B,64.19,4%,National,COSM0091722565B-National,60.68,3.51,Increase,Yes
3168265,LEXA0132425530WXL,102.88,4%,National,LEXA0132425530WXL-National,101.68,1.20,Increase,Yes
3169795,LEXA0552027560T,109.02,4%,National,LEXA0552027560T-National,106.03,2.99,Increase,Yes
3172814,LEXA0551827565E,125.03,4%,National,LEXA0551827565E-National,115.93,9.10,Increase,Yes
3173931,LEXA0102127545WXL,125.93,4%,National,LEXA0102127545WXL-National,119.59,6.34,Increase,Yes
3174969,LION0403031520WXL,128.48,4%,National,LION0403031520WXL-National,122.97,5.51,Increase,Yes


# Step 5: Save DSV

In [ ]:
if save:
    dsv_path = builder.save(df_new_dsv, output_path=dsv_output_path)
    print(f"DSV saved to: {dsv_path}")
else:
    print("[Skipped] DSV save (save=False)")

# Step 6: Update tracker

In [18]:
updater = TrackerUpdater(model.df_current_tests, today_str=date_str)

# Refresh margins from model output
updater.update_margins(df_output)

# Append all new/updated entries
updater.append_entries([
    df_new_nodes_tracker,
    df_low_tracker,
    df_high_tracker,
    df_wm_split_tracker,
    df_margin_tracker,
])

print(f"Tracker: {len(updater.df_tracker)} total rows")
print(f"Expected: {len(model.df_current_tests) + len(df_new_nodes_tracker)}")

display(updater.df_tracker["Final target"].value_counts().sort_index())

2026-03-18 17:01:21,883 [INFO] src.tracker.tracker_updater: Tracker margins updated.
2026-03-18 17:01:24,416 [INFO] src.tracker.tracker_updater: Tracker updated: 3281314 total rows


Tracker: 3281314 total rows
Expected: 3281314


Final target
Added                       1261193
Decreased - margin > 20%     133353
Less sales                    15808
Less sales back               15513
Margin test                  430874
No sales test                290602
Normal strategy              496443
Shipping cost added           31921
Updated                      362193
Updates test                 147431
Wm margin split test          95983
Name: count, dtype: int64

In [ ]:
if save:
    tracker_path = updater.save(output_path=tracker_output_path, backup=tracker_backup)
    print(f"Tracker saved to: {tracker_path}")
else:
    print("[Skipped] Tracker save (save=False)")

# Step 7: [Optional] Upload DSV to hybris

Toggle: `upload_to_hybris` flag in Params cell.

Opens a Chrome browser, signs into hybris backoffice, navigates to the
DSV Prices page, selects the WalmartB2B channel, uploads the CSV file,
and waits for the upload to finish (Status=FINISHED, Result=SUCCESS).

Requires: Selenium, Chrome, `webdriver_manager`, and hybris credentials
on the shared drive.

In [20]:
if upload_to_hybris:
    if not save:
        print("ERROR: save=False but upload_to_hybris=True. Save the DSV first.")
    else:
        from src.dsv.hybris_uploader import HybrisUploader

        print(f"Uploading DSV to hybris: {dsv_path}")
        with HybrisUploader(headless=False) as uploader:
            success = uploader.upload(dsv_path, wait_for_result=True)

        if success:
            print("Hybris upload successful.")
            print("FTP response validation can be run in ~3 hours.")
        else:
            print("Hybris upload failed or timed out. Check hybris manually.")
else:
    print("[Skipped] Hybris upload")

[Skipped] Hybris upload


# Cleanup

In [21]:
loader.close()
print("Done.")

2026-03-18 17:02:46,707 [INFO] src.adapters.google_api_adapter: Google API service connection closed.
2026-03-18 17:02:46,708 [INFO] src.adapters.dw_adapter: Data warehouse adapter closed.


Done.


---

# Post-upload: FTP validation

Run this section **~3 hours after uploading** the DSV via hybris.

Downloads XML response files from Walmart FTP, parses ingestion status,
and generates a summary report. Alerts if failure rate exceeds 1.5%.

In [22]:
from src.dsv.ftp_validator import FTPValidator

validator = FTPValidator(today_str=date_str)
n_files = validator.download_responses()
print(f"Downloaded {n_files} response files")

Connected to FTP server: upload.tires-easy.com


2026-03-18 17:03:25,956 [INFO] src.dsv.ftp_validator: Downloaded 6 response files to G:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Price updates check\2026-03-18\response_files


Downloaded 6 response files


In [23]:
if n_files > 0:
    df_results = validator.parse_responses()
    display(df_results["ingestionStatus"].value_counts())

    report_path = validator.generate_report(df_results)
    print(f"Report saved: {report_path}")
else:
    print("No response files yet. Try again later.")

2026-03-18 17:03:29,405 [INFO] src.dsv.ftp_validator: Parsed 52014 total records from 6 files
2026-03-18 17:03:29,488 [INFO] src.dsv.ftp_validator: Ingestion: 51409 success, 605 failures (1.16% failure rate)


ingestionStatus
SUCCESS       51409
DATA_ERROR      605
Name: count, dtype: int64

2026-03-18 17:03:29,861 [INFO] src.dsv.ftp_validator: Report saved: G:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Price updates check\2026-03-18\NLC Price update response summary 2026-03-18.xlsx


Report saved: G:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Price updates check\2026-03-18\NLC Price update response summary 2026-03-18.xlsx
